In [1]:
import numpy as np
import matplotlib.pyplot as plt

def mi_funcion_sen(vmax, dc, ff, ph, nn, fs):
    xx=dc+vmax*np.sin(2*np.pi*ff*nn/fs+2*np.pi*ph)
    tt=nn/fs
    return(tt,xx)

def espectro_db(xx):
    N=len(xx)
    tr=(1/N)*np.fft.fft(xx)
    modulo_tr_db = 10 * np.log10(2*(np.abs(tr)[:N//2]) ** 2)
    return modulo_tr_db

def ADC (B, Vfs):
    qq=(2*Vfs)/(2**B) # paso de cuantizacion
    pq=qq**2/12 # Potencia teórica del ruido de cuantización
    return (qq,pq)

In [2]:
def cambiar_configuracion(B, kn):
    
    Vfs=2.0 #Volts
    qq,pq=ADC(B,Vfs)
    
    # Ruido analógico
    Pn=kn*pq # potencia del ruido analogico
    sig_ruido=np.sqrt(Pn) #Desviación estándar que necesita la función para generar el ruido
    #snr=10*np.log10(pot_seno/Pn)
    
    #Señal con Ruido
    mu=0 #valor
    yy=np.random.normal(mu,sig_ruido,N)# ruido analógico
    funcion_con_ruido=xx+yy
    
    #Proceso de Cuantizacion
    
    "Cuantizacion"
    xx_q=np.round(funcion_con_ruido/qq)*qq
    nq=xx_q-funcion_con_ruido # ruido por cuantización    
    
    abs_digital_db=espectro_db(xx_q) #Señal con ruido de salida (ruido analógico + ruido de cuantización)
    #abs_digital_db=espectro_db(nq) #Ruido por cuantizacion
    abs_pura_db=espectro_db(xx)
    abs_analogica_db=espectro_db(funcion_con_ruido)
    return(xx_q,funcion_con_ruido,abs_digital_db,abs_pura_db,abs_analogica_db,nq,qq)

In [3]:
def graficador_signal_temp(B,kn): 
    fig=plt.figure(figsize=(10, 5))
    
    # Señales en el dominio Temporal
    plt.plot(tt, xx, ':', label='Señal', color='orange', alpha=0.8) #ruido de la analogica
    plt.plot(tt, xx_q, label='Salida con ruido', color='tab:blue',linestyle='-',linewidth=1.5) #señal con ruido
    plt.plot(tt, funcion_con_ruido, color='green', linestyle=':', marker='o', markersize=1, label='Entrada con Ruido')
    # Formato de la gráfica
    plt.title(f"Señal muestreada por un ADC de {B} bits")
    plt.xlabel("Tiempo [S]")
    plt.ylabel("Amplitud [V]")
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)    
    plt.show()
    plt.close(fig)
    
def graficador_espectros(B, kn):
    # Espectros
    fig=plt.figure(figsize=(12, 5))
    #nq
    plt.plot(frec, abs_digital_db, label='Señal de salida', color='tab:blue', linewidth=1) # Señal Cuantizada
    #xx
    plt.plot(frec, abs_pura_db, ':', label='Señal Pura', color='orange', alpha=0.8) #Señal sin ruido
    #funcion_con_ruido
    plt.plot(frec, abs_analogica_db, ':', label='Señal de entrada', color='darkgreen', alpha=0.6) #señal con ruido
    
    # Formato de la gráfica
    plt.title(f"Señal muestreada por un ADC de {B} bits y kn {kn}")
    plt.xlabel("Frecuencia [Hz]")
    plt.ylabel("Densidad de Potencia [dB]")
    plt.ylim(-80, 10)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)
    
    plt.show()
    plt.close(fig)

def graficador_histograma(nq, qq, B):
    fig=plt.figure(figsize=(8, 5))
    n_bins = 10
    counts, bins, _ = plt.hist(nq, bins=n_bins)
    esperado = len(nq) / n_bins  # altura teórica uniforme
    plt.axvline(-qq/2, color='red', linestyle='--')
    plt.axvline(qq/2, color='red', linestyle='--')
    plt.axhline(esperado, color='red', linestyle='--')
    plt.title(f"Ruido de cuantización para {B} bits - ±$V_R$ = 2.0 V - q = {qq:.3f} V")
    plt.xlabel("Ruido [V]")
    plt.ylabel("Frecuencia")
    #plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
    plt.close(fig)

In [4]:
fs = 1000 # Hz
N = 1000 # muestras
nn=np.arange(0,N)

#Señal de entrada
pot_seno=1
vmax=np.sqrt(pot_seno*2) # (potencial máximo aka apmlitud máxima)
dc=0 # Valor medio (alrededor del que oscila)
df=fs/N # resolucion espectral
k=1 #bin
f_sen=k*df #frecuencia => fk=k*fs/N
ph=0

In [5]:
#Señal pura analógica
tt, xx= mi_funcion_sen(vmax, dc, f_sen, ph, nn, fs)

frec = np.fft.fftfreq(N, 1/fs)[:N//2] #Esto permite graficar en función de la frecuencia y no por los indices k 

In [6]:
xx_q,funcion_con_ruido,abs_digital_db,abs_pura_db,abs_analogica_db,nq,qq=cambiar_configuracion(B=4,kn=1)
#graficador_signal_temp(B=4,kn=1)
#graficador_espectros(B=4,kn=1)
#graficador_histograma(nq, qq, 4)

### Bonus:
#### Encontrar la relación entre la cantidad de B bits del ADC y el SNR de la señal digitalizada.
El SNR es la relación entre la potencia de la señal de interés y la del ruido de fondo. Para poder llegar a la relación de este indicador con los bits del adc es necesario repasar algunas expresiones.<br>
En este proyecto se trabaja con dos potencias principales:
* Potencia de la señal
\begin {equation}
    P_s=\frac{A^2}{2} \tag{1}
\end {equation}
* Potencia del ruido de cuantización
\begin {equation}
    P_q=\frac{q^2}{12} \tag{2}
\end {equation}
<br>
Se define a la amplitud como
\begin {equation}
    A=V_{FS} \tag{3}
\end {equation}
y el paso de cuantización como
\begin {equation}
    q=\frac{2 \cdot V_{FS}}{2^B} \tag{4}
\end {equation}
Si se desea medir el SNR en dB debe implementarse la expresión
\begin {equation}
    SNR_{dB}=10 \cdot log_{10}\left(\frac{P_s}{P_q}\right) \tag{5}
\end {equation}
Reemplazando $(3)$ en $(1)$ y $(4)$ en $(2)$ se obtiene:
\begin {equation}
    P_s=\frac{V_{FS}^2}{2} \tag{6}
\end {equation}
\begin {equation}
    P_q=\frac{\left(\frac{2 \cdot V_{FS}}{2^B}\right)^2}{12} \tag{7}
\end {equation}
Finalmente se sustituyen $(6)$ y $(7)$ en $(5)$:
\begin {equation}
    SNR_{dB}=10\cdot log_{10}\left(\frac{\frac{V_{FS}^2}{2}}{\frac{\left(\frac{2 \cdot V_{FS}}{2^B}\right)^2}{12}}\right) \tag{8}
\end {equation}
Trabajando este arreglo se obtiene que:
\begin {equation}
    SNR_{dB}=10\left(log_{10}\left(\frac{3}{2}\right)+B\cdot log{10}(4)\right)=1,76+6,02\cdot B [dB] \tag{9}
\end {equation}
Esta ecuación es válida si los supuestos de las ecuaciones $(1)$ y $(2)$ y $(3)$se mantienen.<br>
Es posible ver a partir de la fórmula $(9)$ que cada bit que se agregue al ADC resultará en un aumento del SNR en aproximadamente 6.02dB. Esto se debe a que aumentar los bits disminuye el paso de cuantización, y por consecuencia, la potencia del ruido de cuantización. Con esto se demuestra que a menor potencia de ruido, mejor se puede modelar la señal de salida.